In [9]:
#FASE 4 — Task 1 Segmentazione RFM (Recency, Frequency, Monetary)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import os

# Setup
sns.set_theme(style="white")
os.makedirs('output/grafici', exist_ok=True)

# Caricamento dataset pulito
file_path = os.path.join('output', 'dati_puliti', 'cleaned_retail.csv')
df = pd.read_csv(file_path)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Lavoro solo sui clienti identificati (escludiamo anonimi per RFM)
df_rfm = df[df['Customer ID'].notna()].copy()

print(f"Dataset caricato: {df.shape[0]} righe totali")
print(f"Dataset RFM (clienti identificati): {df_rfm.shape[0]} righe")
df_rfm.head()

Dataset caricato: 876436 righe totali
Dataset RFM (clienti identificati): 682244 righe


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,is_anonymous,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0
3,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,30.0
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,False,39.6


In [10]:
# PUNTO 1: Data di riferimento e calcolo Recency
snapshot_date = df_rfm['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Data di riferimento (snapshot): {snapshot_date.date()}")

recency = (
    df_rfm.groupby('Customer ID')['InvoiceDate']
    .max()
    .reset_index()
)
recency.columns = ['Customer ID', 'LastPurchase']
recency['Recency'] = (snapshot_date - recency['LastPurchase']).dt.days

print(f"\nRecency calcolata per {len(recency)} clienti")
print(recency.head())

Data di riferimento (snapshot): 2011-12-10

Recency calcolata per 5682 clienti
   Customer ID        LastPurchase  Recency
0      12346.0 2010-06-28 13:53:00      529
1      12347.0 2011-12-07 15:52:00        2
2      12348.0 2011-04-05 10:47:00      249
3      12349.0 2011-11-21 09:51:00       19
4      12350.0 2011-02-02 16:01:00      310


In [11]:
# PUNTO 2: Calcolo Frequency e Monetary
frequency_monetary = (
    df_rfm.groupby('Customer ID')
    .agg(
        Frequency=('Invoice', 'nunique'),
        Monetary=('TotalPrice', 'sum')
    )
    .reset_index()
)

print(f"Frequency e Monetary calcolati per {len(frequency_monetary)} clienti")
print(frequency_monetary.head())
print(f"\nStatistiche:")
print(frequency_monetary[['Frequency', 'Monetary']].describe())

Frequency e Monetary calcolati per 5682 clienti
   Customer ID  Frequency  Monetary
0      12346.0         11    372.86
1      12347.0          8   3985.81
2      12348.0          4    312.36
3      12349.0          3   2960.24
4      12350.0          1    294.40

Statistiche:
         Frequency       Monetary
count  5682.000000    5682.000000
mean      5.899331    1588.768305
std      11.563258    3936.317678
min       1.000000       1.900000
25%       1.000000     258.627500
50%       3.000000     629.380000
75%       6.000000    1605.477500
max     368.000000  185373.230000


In [12]:
# PUNTO 3.1: Unione delle metriche nel DataFrame RFM finale
#Nota: l'unione delle metriche è il ponte necessario per poter assegnare i punteggi. Non posso dare i voti prima di aver messo tutti i dati nella stessa tabella.
rfm_df = pd.merge(recency[['Customer ID', 'Recency']], frequency_monetary, on='Customer ID')

print(f"Tabella RFM finale creata con successo per {len(rfm_df)} clienti.")
print(rfm_df.head())

Tabella RFM finale creata con successo per 5682 clienti.
   Customer ID  Recency  Frequency  Monetary
0      12346.0      529         11    372.86
1      12347.0        2          8   3985.81
2      12348.0      249          4    312.36
3      12349.0       19          3   2960.24
4      12350.0      310          1    294.40


In [14]:
# PUNTO 3.2: Assegnazione dei punteggi da 1 a 5 con pd.qcut()

# Per la Recency: giorni bassi = voto alto (5)
rfm_df['R_Score'] = pd.qcut(rfm_df['Recency'], q=5, labels=[5, 4, 3, 2, 1])

# Per la Frequency: uso il rank percentuale per evitare l'errore dei duplicati sui quintili
rfm_df['F_Score'] = pd.qcut(rfm_df['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])

# Per il Monetary: valori alti = voto alto (5)
rfm_df['M_Score'] = pd.qcut(rfm_df['Monetary'], q=5, labels=[1, 2, 3, 4, 5])

print("Punteggi R, F, M ottimizzati e calcolati con successo!")
print(rfm_df[['Customer ID', 'Recency', 'R_Score', 'Frequency', 'F_Score', 'Monetary', 'M_Score']].head())

Punteggi R, F, M ottimizzati e calcolati con successo!
   Customer ID  Recency R_Score  Frequency F_Score  Monetary M_Score
0      12346.0      529       1         11       5    372.86       2
1      12347.0        2       5          8       4   3985.81       5
2      12348.0      249       2          4       3    312.36       2
3      12349.0       19       5          3       3   2960.24       5
4      12350.0      310       2          1       1    294.40       2


In [15]:
# PUNTO 4: Creazione del punteggio RFM combinato e segmenti

# 1. Pulizia estetica del Customer ID (rimuove il .0)
rfm_df['Customer ID'] = rfm_df['Customer ID'].astype(int)

# 2. Creazione della stringa di segmento combinata (es. "545")
rfm_df['RFM_Segment'] = (
    rfm_df['R_Score'].astype(str) +
    rfm_df['F_Score'].astype(str) +
    rfm_df['M_Score'].astype(str)
)

# 3. Creazione del punteggio numerico combinato (Somma totale dei voti)
# Questo trasforma i voti in un numero da 3 a 15, utilissimo per le analisi quantitative
rfm_df['RFM_Score_Total'] = rfm_df[['R_Score', 'F_Score', 'M_Score']].astype(int).sum(axis=1)

print("Punteggi e segmenti combinati inseriti con successo!")
print(rfm_df[['Customer ID', 'RFM_Segment', 'RFM_Score_Total']].head())

Punteggi e segmenti combinati inseriti con successo!
   Customer ID RFM_Segment  RFM_Score_Total
0        12346         152                8
1        12347         545               14
2        12348         232                7
3        12349         535               13
4        12350         212                5


In [16]:
# PUNTO 5: Mappatura dei segmenti RFM testuali

# Definisco il dizionario con le regole di mappatura basate su R_Score e F_Score
# (Uso le espressioni regolari per mappare le combinazioni di R e F)
segs = {
    r'[1-2][1-2]': 'Hibernating',      # R basso, F basso: clienti dormienti/persi
    r'[1-2][3-4]': 'At Risk',          # R basso, F medio: clienti fedeli che stiamo perdendo
    r'[1-2]5': 'Can\'t Lose Them',     # R basso, F massimo: erano i migliori, non comprano da tanto
    r'3[1-2]': 'About to Sleep',       # R medio, F basso: tiepidi, rischiano di addormentarsi
    r'33': 'Need Attention',           # R medio, F medio: una via di mezzo, serve un'offerta
    r'[3-4][4-5]': 'Loyal Customers',  # R medio/alto, F alto: comprano spesso e di recente
    r'41': 'Promising',                # R alto, F basso: nuovi clienti, promettenti
    r'51': 'New Customers',            # R massimo, F basso: appena arrivati
    r'[4-5][2-3]': 'Potential Loyalists', # R alto, F medio: clienti recenti che stanno comprando di più
    r'5[4-5]': 'Champions'             # R massimo, F massimo: i nostri clienti migliori in assoluto
}

# Creo la combinazione R + F come stringa temporanea per fare la mappatura
rfm_df['R_F_Group'] = rfm_df['R_Score'].astype(str) + rfm_df['F_Score'].astype(str)

# Applichivo la mappatura usando il dizionario
rfm_df['Segment'] = rfm_df['R_F_Group'].replace(segs, regex=True)

# Rimuovo la colonna temporanea per tenere pulito il DataFrame
rfm_df.drop(columns=['R_F_Group'], inplace=True)

print("Mappatura in segmenti testuali completata!")
print(rfm_df[['Customer ID', 'RFM_Segment', 'Segment']].head())

Mappatura in segmenti testuali completata!
   Customer ID RFM_Segment              Segment
0        12346         152      Can't Lose Them
1        12347         545            Champions
2        12348         232              At Risk
3        12349         535  Potential Loyalists
4        12350         212          Hibernating


In [17]:
# PUNTO 6: Esportazione del dataset RFM finale
rfm_df.to_csv('data/rfm_output.csv', index=False)
print("Tabella RFM finale salvata con successo in 'data/rfm_output.csv'!")

Tabella RFM finale salvata con successo in 'data/rfm_output.csv'!
